# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sohamrathi17/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane asks **"which pages should an editor review first?"**, so the model's job is to produce a *score to sort by*, judged at precision@20 per client (ML-03), not a yes/no label.

**The bar to beat** is my frozen ML-07 rule: score = how many impressions per day a page lost between early and late March, ×1.5 if its CTR is below the median for its position tier. On this slice it reached **precision@20 0.711** with **24.1%** of each client's April lost impressions in its top 20.

**Candidates, fixed before looking at the comparison table:**

| Model | Why it fits |
|---|---|
| **Logistic regression** | Readable: one weight per feature, so I can see what it leans on. Its probability is a ranking score. |
| **Gradient-boosted trees** (classifier) | Can pick up interactions (e.g. age matters differently by position, ML-03) that a linear model misses. |
| **Gradient-boosted trees** (regressor on lost impressions) | Predicts *how much* a page will lose, which is closer to the second metric. |
| **Logistic probability × March drop per day** | The decision score: "how likely is a decline" × "how big is the slide already". Combines the model with the size signal the rule showed matters. |

**Features (16, all knowable on 31 March):**
- **The five from ML-04:** March impressions (log), average position, CTR, age in days, momentum.
- **More March signals:**
  - clicks (log)
  - days with impressions
  - daily volatility
  - position change within March
  - share of impressions on the single biggest day
  - the within-March drop (log)
- **Keyword metadata:** search volume, competition, keyword length, content type, intent.

Excluded as in ML-04: everything from April, the IDs, and the snapshot "last updated" dates.

In [1]:
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# numpy 2 + Apple Accelerate prints spurious "overflow in matmul" warnings inside sklearn; results are unaffected
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sohamrathi17/flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("work/scripts") and os.getcwd() != "/":
        os.chdir("..")
sys.path.insert(0, "work/scripts")

import numpy as np, pandas as pd, sklearn
import lane_data as L   # my shared slice builder, baseline rule and models (work/scripts/lane_data.py)

# Same slice as ML-04 and ML-07: decision moment 31 March 2026, features from March, label from April.
# The token comes from env / Colab Secret / local `hf auth login`, never from a cell.
frame = L.add_drop_feature(L.build_frame("2026-03", "2026-04"))
print(f"{len(frame):,} pages | {frame['client_hash_id'].nunique()} clients | declining in April {frame['is_declining'].mean():.1%}")
print(f"scikit-learn {sklearn.__version__}, seed {L.SEED}")

50,299 pages | 32 clients | declining in April 47.4%
scikit-learn 1.6.1, seed 42


## 2. Split design

**Grouped 5-fold cross-validation by client** (`GroupKFold`, grouped on `client_hash_id`).

- Every page is scored by a model trained on the *other* clients only, so each client is a site the model never saw. That matches how the queue would be used on a new client, and it stops the model from memorising one site's quirks (ML-01 showed a rule that looked good in-sample lose most of its edge on new clients).
- Using all five folds instead of one split matters here: in ML-04 a single 75/25 split put 38% of the test pages in one client. Out-of-fold scores give *every* client a fair score, and the metric is then computed per client exactly as for the baseline.
- The baseline rule needs no training, so it is scored on the same pages, same clients, same metric.

**Note on fold sizes:** the three biggest clients each fill a fold on their own, so three folds are one client each. That is fine because the metric is computed per client anyway, but it shows again how concentrated the slice is.

**What this split does not test:** time. Train and test are both March→April. Whether a model fitted on an earlier month still ranks well on a later one is the ML-09 check.

In [2]:
from sklearn.model_selection import GroupKFold

folds = list(GroupKFold(n_splits=5).split(frame, groups=frame["client_hash_id"]))
for i, (tr, te) in enumerate(folds, 1):
    print(f"fold {i}: test {len(te):>6,} pages from {frame.iloc[te]['client_hash_id'].nunique():>2} clients "
          f"| train {len(tr):>6,} pages | test declining rate {frame.iloc[te]['is_declining'].mean():.1%}")
print("\nNo client appears in both train and test of any fold:",
      all(set(frame.iloc[tr]['client_hash_id']).isdisjoint(frame.iloc[te]['client_hash_id']) for tr, te in folds))

fold 1: test 14,568 pages from  1 clients | train 35,731 pages | test declining rate 37.6%
fold 2: test  9,721 pages from  1 clients | train 40,578 pages | test declining rate 65.0%
fold 3: test  9,191 pages from  1 clients | train 41,108 pages | test declining rate 49.5%
fold 4: test  8,410 pages from 15 clients | train 41,889 pages | test declining rate 56.8%
fold 5: test  8,409 pages from 14 clients | train 41,890 pages | test declining rate 32.5%

No client appears in both train and test of any fold: True


## 3. Train + compare vs my baseline

Same 50,299 pages, same 22 evaluated clients (those with at least 20 pages), same metrics as ML-07. Every model score below is out-of-fold. ROC-AUC (how well a score separates decliners from non-decliners overall, 0.5 = coin flip) is shown for the classifiers as an extra.

In [3]:
from sklearn.metrics import roc_auc_score

daily_drop = L.drop_per_day(frame)
p_logit = L.out_of_fold(frame, L.make_logistic, "is_declining")
p_boost = L.out_of_fold(frame, L.make_boosted_classifier, "is_declining")
frame["log_lost"] = np.log1p(frame["impressions_lost"])           # regression target only
e_loss  = L.out_of_fold(frame, L.make_boosted_regressor, "log_lost", proba=False)

scores = {
    "random order (expected)":            None,
    "ML-07 baseline rule":                L.baseline_score(frame),
    "logistic regression":                p_logit,
    "boosted trees (classifier)":         p_boost,
    "boosted trees (expected loss)":      e_loss,
    "logistic prob x March drop/day":     p_logit * (1 + daily_drop),
}
rows = []
for name, s in scores.items():
    if s is None:
        base = L.per_client_metrics(frame, np.zeros(len(frame)))
        rows.append({"ranking": name, "precision@20": base["base_rate"].mean(), "loss captured@20": 20 / base["pages"]})
        rows[-1]["loss captured@20"] = (20 / base["pages"]).clip(upper=1).mean()
        continue
    m = L.summarize(frame, s)
    rows.append({"ranking": name, "precision@20": m["precision@20"], "loss captured@20": m["loss_captured@20"]})
table = pd.DataFrame(rows).set_index("ranking")
table["ROC-AUC"] = [np.nan, np.nan, roc_auc_score(frame["is_declining"], p_logit),
                    roc_auc_score(frame["is_declining"], p_boost), np.nan, np.nan]
print(f"Base rate (average of per-client declining rates): {L.summarize(frame, p_logit)['base_rate']:.3f}\n")
table.round(3)

Base rate (average of per-client declining rates): 0.489



,precision@20,loss captured@20,ROC-AUC
ranking,,,
random order (expected),0.489,0.104,NaN
ML-07 baseline rule,0.711,0.241,NaN
logistic regression,0.750,0.197,0.633
boosted trees (classifier),0.734,0.174,0.653
boosted trees (expected loss),0.691,0.264,NaN
logistic prob x March drop/day,0.718,0.242,NaN


Random order's precision equals the base rate, and its expected loss captured is just 20 ÷ the client's page count. Both rows are there so every number has its naive comparison next to it.

Now client by client, for the two strongest candidates against the baseline:

In [4]:
pc_base  = L.per_client_metrics(frame, L.baseline_score(frame)).set_index("client")
pc_logit = L.per_client_metrics(frame, p_logit).set_index("client")
pc_comb  = L.per_client_metrics(frame, p_logit * (1 + daily_drop)).set_index("client")

for name, pc in [("logistic regression", pc_logit), ("logistic prob x drop", pc_comb)]:
    diff_p = pc["precision_at_k"] - pc_base["precision_at_k"]
    diff_l = pc["loss_captured_at_k"] - pc_base["loss_captured_at_k"]
    print(f"{name:22} precision@20: better in {(diff_p > 0).sum():>2}, same in {(diff_p == 0).sum():>2}, worse in {(diff_p < 0).sum():>2} of {len(pc)} clients"
          f" | loss captured: better in {(diff_l > 0.001).sum():>2}, worse in {(diff_l < -0.001).sum():>2}")

metrics = {
    "assignment": "ML-08 model vs baseline",
    "slice": "decision 2026-03-31, features March 2026, label April 2026 (same as ML-04/ML-07)",
    "split": "GroupKFold(5) by client, out-of-fold scores; metrics per client (>= 20 pages), then averaged",
    "seed": L.SEED, "sklearn_version": sklearn.__version__,
    "n_pages": int(len(frame)), "n_clients": int(frame["client_hash_id"].nunique()),
    "results": {k: {c: (None if pd.isna(v) else round(float(v), 4)) for c, v in r.items()} for k, r in table.to_dict("index").items()},
}
os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/model_metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)
print("\nWrote work/outputs/model_metrics.json")

logistic regression    precision@20: better in 10, same in  7, worse in  5 of 22 clients | loss captured: better in  2, worse in 20
logistic prob x drop   precision@20: better in  5, same in 14, worse in  3 of 22 clients | loss captured: better in  9, worse in  7

Wrote work/outputs/model_metrics.json


**What the comparison shows (out-of-fold, 22 clients):**

- **No model beats the rule on both metrics at once.** That is the honest headline.
- **Logistic regression** is the precision winner: its top 20 per client contain more true decliners than the rule's. But it catches *less* of the lost impressions, because it happily ranks small pages that are likely to dip.
- **The boosted-tree classifier** separates decliners best overall (highest ROC-AUC) and also beats the rule on precision, but its top 20 is not better than the logistic model's and it catches the least loss of all. A higher AUC across all pages is not the same thing as a better short list.
- **The expected-loss regressor** trades in the other direction: it catches a bit more of the lost impressions than the rule (about 26% vs 24%) but with lower precision (about 0.69 vs 0.71).
- **The combined score** lands almost exactly on top of the rule: slightly higher precision, the same loss captured.

My reading: in this slice, *the size of the slide already visible in March* carries most of the signal, and the hand rule already uses it. The learned models confirm that, add a little precision, and do not find a big hidden pattern on top. For the queue I will carry forward the **combined score (logistic probability × March drop)**, because it keeps the rule's loss coverage while adding the model's precision and a readable set of weights. I chose it after seeing this table, so ML-09 re-tests it on a *later* month it was not tuned on.

## 4. Errors and interpretation

**What does the logistic model lean on?** Standardised weights (positive = more likely to decline). Because the inputs are scaled, the sizes are comparable.

In [5]:
final_logit = L.make_logistic().fit(frame[L.MODEL_FEATURES], frame["is_declining"])
names = final_logit[0].get_feature_names_out()
weights = pd.Series(final_logit[-1].coef_[0], index=[n.split("__")[-1] for n in names]).sort_values()
print("Most 'decline' weight:\n" + weights.tail(5).round(3).to_string())
print("\nMost 'holds up' weight:\n" + weights.head(5).round(3).to_string())

Most 'decline' weight:
content_type_comparison article    0.280
top_day_share                      0.310
main_intent_unknown                0.327
log_imp                            0.475
log_drop                           0.539

Most 'holds up' weight:
log_clicks                     -0.677
momentum                       -0.228
content_type_keyword article   -0.207
imp_cv                         -0.195
main_intent_informational      -0.185


**Where is it wrong?** Below: the precision@20 of the combined score by client size, and the April outcome of its top-20 picks by March momentum.

In [6]:
comb = p_logit * (1 + daily_drop)
frame["comb_score"] = comb
frame["comb_rank"] = frame.groupby("client_hash_id")["comb_score"].rank(ascending=False, method="first")
top = frame[frame["comb_rank"] <= 20].copy()

pc_comb["size"] = pd.cut(pc_comb["pages"], [0, 100, 1000, np.inf], labels=["20-100 pages", "101-1,000", "1,000+"])
print("Precision@20 of the combined score by client size:")
print(pc_comb.groupby("size", observed=True).agg(clients=("pages", "size"), precision_at_20=("precision_at_k", "mean"),
      base_rate=("base_rate", "mean")).round(3).to_string())

top["momentum_bucket"] = pd.cut(top["momentum"], [0, 0.5, 0.8, 0.95, 1.05, np.inf],
                                labels=["< 0.5", "0.5-0.8", "0.8-0.95", "0.95-1.05", "> 1.05"])
print("\nTop-20 picks (all clients) by March momentum: how many, and how many really declined:")
print(top.groupby("momentum_bucket", observed=True)["is_declining"].agg(picks="size", declined="mean").round(3).to_string())

Precision@20 of the combined score by client size:
              clients  precision_at_20  base_rate
size                                             
20-100 pages        3            0.250      0.256
101-1,000          11            0.791      0.600
1,000+              8            0.794      0.423

Top-20 picks (all clients) by March momentum: how many, and how many really declined:
                 picks  declined
momentum_bucket                 
< 0.5              167     0.784
0.5-0.8            243     0.679
0.8-0.95            47     0.553
0.95-1.05            1     1.000
> 1.05              47     0.170


**Three concrete misses:** the highest-scored pages that did *not* decline in April.

In [7]:
misses = frame[frame["is_declining"] == 0].sort_values("comb_score", ascending=False).head(3)
show = misses.assign(
    march_per_day=(misses["imp"] / 31).round(0),
    early_march_per_day=misses["imp_early_per_day"].round(0),
    late_march_per_day=misses["imp_late_per_day"].round(0),
    april_per_day=(misses["imp_next"] / 30).round(0),
    model_prob=p_logit[misses.index].round(3),
)[["early_march_per_day", "late_march_per_day", "march_per_day", "april_per_day", "avg_position", "top_day_share", "model_prob"]]
show.index = ["miss 1", "miss 2", "miss 3"]
show.round(3)

,early_march_per_day,late_march_per_day,march_per_day,april_per_day,avg_position,top_day_share,model_prob
miss 1,3104.0,537.0,1862.0,1990.0,6.019,0.093,0.989
miss 2,2672.0,484.0,1613.0,1501.0,8.992,0.115,0.943
miss 3,2956.0,1212.0,2112.0,1887.0,36.334,0.065,0.897


**Reading the errors:**

- **The model leans on things that make sense.** The biggest "decline" weights are the size of the March drop, total March impressions, and a big share of the month landing on one day (a spike that fades). The biggest "holds up" weights are clicks and momentum. Impressions and clicks pulling in opposite directions works like a CTR signal: a page with many impressions but few clicks is more likely to decline, which matches the CTR-vs-position check in ML-07. None of the weights is suspiciously large, and no April column is in the inputs, so nothing points to leakage.
- **The misses share one shape.** All three were busy in early March, quieter late in March, and then held or bounced back in April. The April label compares April to the *whole* March average, so a page whose late-March dip was temporary does not count as declining. A score built on "the slide in March" will always flag some of these. Telling a real slide from a spike that fades needs a longer history (e.g. February), which is a feature idea for the capstone.
- **Small clients get no lift.** For the 3 clients with 20 to 100 pages, precision@20 is no better than their base rate. With so few pages, a top 20 is a big share of the whole site, and one or two picks swing the number by 5 to 10 points. The score is only useful for clients with a few hundred pages or more.
- **Careful wording:** this is decision-support. The scores rank pages that *look* worth reviewing first, based on patterns observed in one March→April window. They do not say why a page declines, or that reviewing it will help.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled: markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.